In [0]:
print("Hello Apex Retail")

Hello Apex Retail


In [0]:
base_path = "/Volumes/workspace/apex_retail/raw_landing_zone"

df_test = spark.read.csv(
    f"{base_path}/historical_data/customer/customer_historical.csv",
    header=True,
    inferSchema=False   # keeps every column as text/string, as Phase 1 requires
)

df_test.printSchema()
df_test.show(5)
print("Row count:", df_test.count())

root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: string (nullable = true)
 |-- churned: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- number_of_children: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+-----------+---+------+--------------+---------------+----------------+-------+--------------+------------------+---------------+-------------+-----------------+-------------+--------------+
|customer_id|age|gender|income_bracket|loyalty_program|membership_years|churned|marital_status|number_of_children|education_level|   occupation|customer_zip_code|customer_city|custo

In [0]:
datasets = ["customer", "product", "sales"]

raw_dfs = {}

for ds in datasets:
    # Historical path pattern: historical_data/{ds}/{ds}_historical.csv
    hist_path = f"{base_path}/historical_data/{ds}/{ds}_historical.csv"
    df_hist = spark.read.csv(hist_path, header=True, inferSchema=False)
    raw_dfs[f"{ds}_historical"] = df_hist
    print(f"{ds}_historical: {df_hist.count()} rows, {len(df_hist.columns)} columns")

    # Incremental path pattern: incremental_data/{ds}_incremental/{ds}_incremental.csv
    incr_path = f"{base_path}/incremental_data/{ds}_incremental/{ds}_incremental.csv"
    df_incr = spark.read.csv(incr_path, header=True, inferSchema=False)
    raw_dfs[f"{ds}_incremental"] = df_incr
    print(f"{ds}_incremental: {df_incr.count()} rows, {len(df_incr.columns)} columns")

customer_historical: 1052 rows, 14 columns
customer_incremental: 1053 rows, 19 columns
product_historical: 1043 rows, 16 columns
product_incremental: 1041 rows, 17 columns
sales_historical: 1002 rows, 19 columns
sales_incremental: 1000 rows, 19 columns


In [0]:
print("=== customer_historical schema ===")
raw_dfs["customer_historical"].printSchema()

print("=== customer_incremental schema ===")
raw_dfs["customer_incremental"].printSchema()

print("=== product_historical schema ===")
raw_dfs["product_historical"].printSchema()

print("=== product_incremental schema ===")
raw_dfs["product_incremental"].printSchema()

=== customer_historical schema ===
root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: string (nullable = true)
 |-- churned: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- number_of_children: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

=== customer_incremental schema ===
root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: string (nullable = true)
 |-- churned: string (nullable = true)
 

In [0]:
# Official column lists per the assignment's Section 2.1 schema
official_columns = {
    "customer": ["customer_id", "age", "gender", "income_bracket", "loyalty_program",
                 "membership_years", "churned", "marital_status", "number_of_children",
                 "education_level", "occupation", "customer_zip_code", "customer_city",
                 "customer_state"],
    "product": ["product_id", "product_name", "product_brand", "product_category",
                "product_rating", "product_review_count", "product_stock",
                "product_return_rate", "product_size", "product_weight", "product_color",
                "product_material", "product_manufacture_date", "product_expiry_date",
                "product_shelf_life", "unit_price"],
    "sales": ["transaction_id", "transaction_date", "customer_id", "product_id", "quantity",
              "unit_price", "discount_applied", "payment_method", "store_location",
              "transaction_hour", "day_of_week", "week_of_year", "month_of_year",
              "total_sales", "promotion_id", "promotion_type", "holiday_season",
              "season", "weekend"]
}

datasets = ["customer", "product", "sales"]
raw_dfs = {}

for ds in datasets:
    cols = official_columns[ds]

    hist_path = f"{base_path}/historical_data/{ds}/{ds}_historical.csv"
    df_hist = spark.read.csv(hist_path, header=True, inferSchema=False).select(*cols)
    raw_dfs[f"{ds}_historical"] = df_hist
    print(f"{ds}_historical: {df_hist.count()} rows, {len(df_hist.columns)} columns")

    incr_path = f"{base_path}/incremental_data/{ds}_incremental/{ds}_incremental.csv"
    df_incr = spark.read.csv(incr_path, header=True, inferSchema=False).select(*cols)
    raw_dfs[f"{ds}_incremental"] = df_incr
    print(f"{ds}_incremental: {df_incr.count()} rows, {len(df_incr.columns)} columns")

customer_historical: 1052 rows, 14 columns
customer_incremental: 1053 rows, 14 columns
product_historical: 1043 rows, 16 columns
product_incremental: 1041 rows, 16 columns
sales_historical: 1002 rows, 19 columns
sales_incremental: 1000 rows, 19 columns


In [0]:
for key, df in raw_dfs.items():
    ds, load = key.rsplit("_", 1)  # splits "customer_historical" -> ("customer", "historical")
    output_path = f"{base_path}/raw/{ds}/{load}/"
    df.write.mode("overwrite").csv(output_path, header=True)
    print(f"Written: {output_path}")

Written: /Volumes/workspace/apex_retail/raw_landing_zone/raw/customer/historical/
Written: /Volumes/workspace/apex_retail/raw_landing_zone/raw/customer/incremental/
Written: /Volumes/workspace/apex_retail/raw_landing_zone/raw/product/historical/
Written: /Volumes/workspace/apex_retail/raw_landing_zone/raw/product/incremental/
Written: /Volumes/workspace/apex_retail/raw_landing_zone/raw/sales/historical/
Written: /Volumes/workspace/apex_retail/raw_landing_zone/raw/sales/incremental/
